# Logging & Debugging Practices in Python

This notebook covers best practices for logging and debugging in Python, including:
- Python's built-in `logging` module
- Log levels and formatting
- File and console handlers
- Structured logging (JSON)
- Debugging with `pdb` and `breakpoint()`
- Using `traceback` and exception handling
- Decorators for logging
- Context managers for debugging


## 1. Basic Logging Setup

In [1]:
import logging

# Basic configuration
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

logger = logging.getLogger(__name__)

# All five log levels
logger.debug('DEBUG: Detailed info, typically of interest only when diagnosing problems')
logger.info('INFO: Confirmation that things are working as expected')
logger.warning('WARNING: Something unexpected happened, but software still works')
logger.error('ERROR: A serious problem — software has not been able to perform a function')
logger.critical('CRITICAL: A serious error, program itself may be unable to continue running')

ERROR:__main__:ERROR: A serious problem — software has not been able to perform a function
CRITICAL:__main__:CRITICAL: A serious error, program itself may be unable to continue running


## 2. Log Levels Explained

In [3]:
import logging

# Log level hierarchy (lowest to highest)
levels = {
    'DEBUG':    logging.DEBUG,     # 10
    'INFO':     logging.INFO,      # 20
    'WARNING':  logging.WARNING,   # 30
    'ERROR':    logging.ERROR,     # 40
    'CRITICAL': logging.CRITICAL   # 50
}

for name, value in levels.items():
    print(f'{name:10s} = {value}')

print()
print('Setting level to WARNING means only WARNING, ERROR, and CRITICAL are shown.')

DEBUG      = 10
INFO       = 20
WARNING    = 30
ERROR      = 40
CRITICAL   = 50

Setting level to WARNING means only WARNING, ERROR, and CRITICAL are shown.


## 3. Named Loggers (Best Practice)

In [13]:
import logging

# Always use named loggers — never use root logger in production
# Use __name__ so the logger name reflects the module hierarchy

def get_logger(name: str) -> logging.Logger:
    """Create and return a configured logger."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)

    # Avoid duplicate handlers if called multiple times
    if not logger.handlers:
        handler = logging.StreamHandler()
        formatter = logging.Formatter(
            '%(asctime)s | %(name)s | %(levelname)-8s | %(message)s',
            datefmt='%Y-%m-%d %H:%M:%S'
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)
        logger.propagate = False  # Don't pass to root logger

    return logger


# Simulating module-level loggers
db_logger     = get_logger('myapp.database')
api_logger    = get_logger('myapp.api')
service_logger= get_logger('myapp.service')

db_logger.info('Connected to database successfully')
api_logger.warning('API rate limit approaching: 90% used')
service_logger.error('Payment service timeout after 30s')

2026-04-02 04:53:13 | myapp.database | INFO     | Connected to database successfully
2026-04-02 04:53:13 | myapp.api | WARNING  | API rate limit approaching: 90% used
2026-04-02 04:53:13 | myapp.service | ERROR    | Payment service timeout after 30s


## 4. File Handler — Logging to a File

In [14]:
import logging
from logging.handlers import RotatingFileHandler
import os

log_file = 'app.log'

file_logger = logging.getLogger('file_demo')
file_logger.setLevel(logging.DEBUG)

if not file_logger.handlers:
    # RotatingFileHandler: creates new log file when size limit is hit
    file_handler = RotatingFileHandler(
        log_file,
        maxBytes=1_000_000,   # 1 MB per file
        backupCount=3         # Keep last 3 files
    )
    file_handler.setLevel(logging.DEBUG)

    # Console handler only for WARNING and above
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.WARNING)

    fmt = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(fmt)
    console_handler.setFormatter(fmt)

    file_logger.addHandler(file_handler)
    file_logger.addHandler(console_handler)
    file_logger.propagate = False

file_logger.debug('This goes to the file only')
file_logger.info('This also goes to the file only')
file_logger.warning('This goes to BOTH file and console')
file_logger.error('This also goes to both')

# Verify file exists
if os.path.exists(log_file):
    with open(log_file) as f:
        print('=== Contents of app.log ===')
        print(f.read())

2026-04-02 04:53:16,624 - WARNING - This goes to BOTH file and console
2026-04-02 04:53:16,627 - ERROR - This also goes to both


=== Contents of app.log ===
2026-04-02 04:10:27,976 - DEBUG - This goes to the file only
2026-04-02 04:10:27,977 - INFO - This also goes to the file only
2026-04-02 04:10:27,977 - WARNING - This goes to BOTH file and console
2026-04-02 04:10:27,979 - ERROR - This also goes to both
2026-04-02 04:53:16,623 - DEBUG - This goes to the file only
2026-04-02 04:53:16,623 - INFO - This also goes to the file only
2026-04-02 04:53:16,624 - WARNING - This goes to BOTH file and console
2026-04-02 04:53:16,627 - ERROR - This also goes to both



## 5. Structured Logging with JSON

In [17]:
import logging
import json
from datetime import datetime, timezone

# Clear any existing 'json_demo' logger from previous runs
logger_to_reset = logging.getLogger('json_demo')
logger_to_reset.handlers.clear()

class JSONFormatter(logging.Formatter):
    """Custom formatter that outputs log records as clean JSON."""

    # All standard LogRecord fields — we handle these manually or skip them
    _STANDARD_FIELDS = {
        'name', 'msg', 'args', 'levelname', 'levelno', 'pathname',
        'filename', 'module', 'exc_info', 'exc_text', 'stack_info',
        'lineno', 'funcName', 'created', 'msecs', 'relativeCreated',
        'thread', 'threadName', 'processName', 'process', 'taskName',
        'message',
    }

    def format(self, record: logging.LogRecord) -> str:
        log_entry = {
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'level':     record.levelname,
            'logger':    record.name,
            'message':   record.getMessage(),
            'module':    record.module,
            'line':      record.lineno,
        }

        # Handle exception info separately — exc_info is a tuple of types,
        # which are NOT JSON serializable, so we format it to a string first
        if record.exc_info:
            log_entry['exception'] = self.formatException(record.exc_info)

        # Only include user-supplied extra fields, skip all standard ones
        for key, value in record.__dict__.items():
            if key in self._STANDARD_FIELDS or key.startswith('_'):
                continue
            try:
                json.dumps(value)  # Probe serializability
                log_entry[key] = value
            except (TypeError, ValueError):
                log_entry[key] = repr(value)  # Safe fallback

        return json.dumps(log_entry, indent=None)


json_logger = logging.getLogger('json_demo')
json_logger.setLevel(logging.DEBUG)
json_logger.propagate = False

handler = logging.StreamHandler()
handler.setFormatter(JSONFormatter())
json_logger.addHandler(handler)

# Log with extra structured fields
json_logger.info('User logged in', extra={'user_id': 42, 'ip': '192.168.1.1'})
json_logger.warning('Payment failed', extra={'order_id': 'ORD-999', 'amount': 199.99})

try:
    result = 10 / 0
except ZeroDivisionError:
    json_logger.error('Division error occurred', exc_info=True)

{"timestamp": "2026-04-02T04:55:30.971491+00:00", "level": "INFO", "logger": "json_demo", "message": "User logged in", "module": "819596113", "line": 58, "user_id": 42, "ip": "192.168.1.1"}
{"timestamp": "2026-04-02T04:55:30.979544+00:00", "level": "WARNING", "logger": "json_demo", "message": "Payment failed", "module": "819596113", "line": 59, "order_id": "ORD-999", "amount": 199.99}
{"timestamp": "2026-04-02T04:55:30.983534+00:00", "level": "ERROR", "logger": "json_demo", "message": "Division error occurred", "module": "819596113", "line": 64, "exception": "Traceback (most recent call last):\n  File \"/tmp/ipykernel_7028/819596113.py\", line 62, in <cell line: 0>\n    result = 10 / 0\n             ~~~^~~\nZeroDivisionError: division by zero"}


## 6. Logging Decorator

In [19]:
import logging
import functools
import time

# Remove stale handlers from previous runs
dec_logger = logging.getLogger('decorator_demo')
dec_logger.handlers.clear()
dec_logger.setLevel(logging.DEBUG)
dec_logger.propagate = False  # Don't pass to Jupyter's root logger

handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
dec_logger.addHandler(handler)

logger = dec_logger


def log_call(func):
    """Decorator: logs function entry, exit, return value, and elapsed time."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.debug(f'→ Calling {func.__name__}() | args={args}, kwargs={kwargs}')
        start = time.perf_counter()
        try:
            result = func(*args, **kwargs)
            elapsed = time.perf_counter() - start
            logger.debug(f'← {func.__name__}() returned {result!r} in {elapsed:.4f}s')
            return result
        except Exception as e:
            elapsed = time.perf_counter() - start
            logger.error(f'✗ {func.__name__}() raised {type(e).__name__}: {e} after {elapsed:.4f}s')
            raise
    return wrapper


@log_call
def add(a, b):
    return a + b

@log_call
def divide(a, b):
    return a / b

@log_call
def slow_operation(n):
    time.sleep(0.1)
    return n * 2

add(3, 7)
slow_operation(5)

try:
    divide(10, 0)
except ZeroDivisionError:
    pass  # Already logged by decorator

2026-04-02 04:57:06,349 - DEBUG - → Calling add() | args=(3, 7), kwargs={}
2026-04-02 04:57:06,351 - DEBUG - ← add() returned 10 in 0.0000s
2026-04-02 04:57:06,354 - DEBUG - → Calling slow_operation() | args=(5,), kwargs={}
2026-04-02 04:57:06,461 - DEBUG - ← slow_operation() returned 10 in 0.1040s
2026-04-02 04:57:06,465 - DEBUG - → Calling divide() | args=(10, 0), kwargs={}
2026-04-02 04:57:06,472 - ERROR - ✗ divide() raised ZeroDivisionError: division by zero after 0.0000s


## 7. Exception Handling & Traceback Logging

In [21]:
import logging
import traceback
import sys

# Clean setup — route to stdout so print() and logging stay in order
exc_logger = logging.getLogger('exception_demo')
exc_logger.handlers.clear()
exc_logger.setLevel(logging.DEBUG)
exc_logger.propagate = False

handler = logging.StreamHandler(sys.stdout)  # stdout instead of stderr
handler.setFormatter(logging.Formatter('%(levelname)s - %(message)s'))
exc_logger.addHandler(handler)

logger = exc_logger


# ✅ Best practice: always use exc_info=True or logger.exception()
def process_data(data):
    try:
        result = [int(x) for x in data]
        return sum(result)
    except ValueError as e:
        logger.exception('Failed to process data — invalid value found')
        return None


# ❌ Bad practice: silently catching exceptions
def bad_process(data):
    try:
        return sum(int(x) for x in data)
    except Exception:
        pass  # Swallowed! No trace, no log. Avoid this!


# ✅ Custom exception with context
class DataProcessingError(Exception):
    def __init__(self, message, data=None):
        super().__init__(message)
        self.data = data

def strict_process(data):
    try:
        return [int(x) for x in data]
    except ValueError as e:
        raise DataProcessingError(f'Cannot convert data to int: {e}', data=data) from e


print('--- Good exception handling ---')
process_data(['1', '2', 'abc', '4'])

print()
print('--- Manual traceback capture ---')
try:
    strict_process(['10', 'bad'])
except DataProcessingError as e:
    logger.error('DataProcessingError: %s | data=%s', e, e.data)
    print(traceback.format_exc())

--- Good exception handling ---
ERROR - Failed to process data — invalid value found
Traceback (most recent call last):
  File "/tmp/ipykernel_7028/4267758056.py", line 21, in process_data
    result = [int(x) for x in data]
              ^^^^^^
ValueError: invalid literal for int() with base 10: 'abc'

--- Manual traceback capture ---
ERROR - DataProcessingError: Cannot convert data to int: invalid literal for int() with base 10: 'bad' | data=['10', 'bad']
Traceback (most recent call last):
  File "/tmp/ipykernel_7028/4267758056.py", line 44, in strict_process
    return [int(x) for x in data]
            ^^^^^^
ValueError: invalid literal for int() with base 10: 'bad'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/tmp/ipykernel_7028/4267758056.py", line 55, in <cell line: 0>
    strict_process(['10', 'bad'])
  File "/tmp/ipykernel_7028/4267758056.py", line 46, in strict_process
    raise DataProcessingError(f'Cannot 

## 8. Debug with `breakpoint()` and `pdb`

In [9]:
# pdb (Python Debugger) — interactive debugging
# In a script, insert:  breakpoint()  or  import pdb; pdb.set_trace()
# Common pdb commands:
#   n  - next line
#   s  - step into function
#   c  - continue
#   q  - quit
#   p <expr>  - print expression
#   l  - list source code
#   bt - backtrace / call stack
#   u/d - up/down stack frame

# ---- Demo: post-mortem debugging ----
import pdb
import sys

def buggy_function(items):
    total = 0
    for item in items:
        total += item['value']  # Will fail if key is missing
    return total

data = [
    {'value': 10},
    {'value': 20},
    {'val': 30},  # Bug: wrong key!
]

try:
    result = buggy_function(data)
except KeyError as e:
    print(f'KeyError caught: {e}')
    print('In a real script, pdb.post_mortem() would launch interactive debugger here')
    # Uncomment to use interactively in a terminal:
    # pdb.post_mortem()

print('\nTip: Use breakpoint() anywhere in your code to pause execution.')
print('Set PYTHONBREAKPOINT=0 env var to disable all breakpoints in production.')

KeyError caught: 'value'
In a real script, pdb.post_mortem() would launch interactive debugger here

Tip: Use breakpoint() anywhere in your code to pause execution.
Set PYTHONBREAKPOINT=0 env var to disable all breakpoints in production.


## 9. Using `assert` for Debugging

In [10]:
# assert is useful during development to catch invalid states early
# ⚠️ NEVER use assert for input validation in production — it can be disabled with python -O

def calculate_discount(price: float, discount_pct: float) -> float:
    assert 0 <= discount_pct <= 100, f'Discount must be 0-100, got {discount_pct}'
    assert price >= 0, f'Price must be non-negative, got {price}'
    return price * (1 - discount_pct / 100)

# Valid call
print(calculate_discount(100.0, 20))  # 80.0

# Invalid call — AssertionError
try:
    calculate_discount(100.0, 150)  # 150% discount makes no sense
except AssertionError as e:
    print(f'AssertionError: {e}')

# For production validation, raise proper exceptions instead:
def calculate_discount_safe(price: float, discount_pct: float) -> float:
    if not (0 <= discount_pct <= 100):
        raise ValueError(f'Discount must be 0-100, got {discount_pct}')
    if price < 0:
        raise ValueError(f'Price must be non-negative, got {price}')
    return price * (1 - discount_pct / 100)

80.0
AssertionError: Discount must be 0-100, got 150


## 10. Context Manager for Temporary Debug Logging

In [11]:
import logging
from contextlib import contextmanager


@contextmanager
def verbose_logging(logger_name: str, level=logging.DEBUG):
    """Temporarily lower log level for a specific section of code."""
    logger = logging.getLogger(logger_name)
    original_level = logger.level
    logger.setLevel(level)
    print(f'[DebugCtx] Enabled verbose logging for "{logger_name}"')
    try:
        yield logger
    finally:
        logger.setLevel(original_level)
        print(f'[DebugCtx] Restored log level for "{logger_name}"')


app_logger = logging.getLogger('myapp')
app_logger.setLevel(logging.WARNING)  # Normally only warnings
app_logger.addHandler(logging.StreamHandler())
app_logger.propagate = False

app_logger.debug('This will NOT appear (level is WARNING)')  # suppressed
app_logger.warning('This WILL appear')

print()
with verbose_logging('myapp') as log:
    log.debug('Now this DEBUG message DOES appear')
    log.info('And this INFO message too')

print()
app_logger.debug('Back to normal — this is suppressed again')
app_logger.warning('Only warnings get through again')

This WILL appear
Now this DEBUG message DOES appear
And this INFO message too
Only warnings get through again



[DebugCtx] Enabled verbose logging for "myapp"
[DebugCtx] Restored log level for "myapp"



## 11. Complete Real-World Example

In [12]:
import logging
import json
import time
import functools
from typing import Optional


# ---- Logger setup ----
def setup_logger(name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)
    if not logger.handlers:
        h = logging.StreamHandler()
        h.setFormatter(logging.Formatter(
            '%(asctime)s | %(name)-20s | %(levelname)-8s | %(message)s',
            datefmt='%H:%M:%S'
        ))
        logger.addHandler(h)
        logger.propagate = False
    return logger


# ---- Retry decorator with logging ----
def retry(retries=3, delay=0.5, exceptions=(Exception,)):
    def decorator(func):
        logger = setup_logger(f'retry.{func.__name__}')
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, retries + 1):
                try:
                    result = func(*args, **kwargs)
                    if attempt > 1:
                        logger.info(f'Succeeded on attempt {attempt}')
                    return result
                except exceptions as e:
                    logger.warning(f'Attempt {attempt}/{retries} failed: {e}')
                    if attempt < retries:
                        time.sleep(delay)
                    else:
                        logger.error(f'All {retries} attempts failed. Giving up.')
                        raise
        return wrapper
    return decorator


# ---- Simulated service ----
call_count = 0

@retry(retries=3, delay=0.1, exceptions=(ConnectionError,))
def fetch_user(user_id: int) -> dict:
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError('Network unreachable')
    return {'id': user_id, 'name': 'Alice', 'email': 'alice@example.com'}


svc_logger = setup_logger('user_service')
svc_logger.info('Fetching user 42...')

try:
    user = fetch_user(42)
    svc_logger.info(f'Got user: {user}')
except ConnectionError:
    svc_logger.critical('Could not fetch user after retries')

04:11:48 | user_service         | INFO     | Fetching user 42...
04:11:48 | retry.fetch_user     | WARNING  | Attempt 1/3 failed: Network unreachable
04:11:48 | retry.fetch_user     | WARNING  | Attempt 2/3 failed: Network unreachable
04:11:48 | retry.fetch_user     | INFO     | Succeeded on attempt 3
04:11:48 | user_service         | INFO     | Got user: {'id': 42, 'name': 'Alice', 'email': 'alice@example.com'}


## Summary

| Practice | Why it matters |
|---|---|
| Use named loggers (`__name__`) | Isolate log output per module |
| Use `RotatingFileHandler` | Prevent log files from growing forever |
| Use `logger.exception()` in except blocks | Always captures full traceback |
| Structured (JSON) logging | Machine-parseable; works with log aggregators |
| Log decorators | Consistent tracing without cluttering business logic |
| Never swallow exceptions silently | Always log before suppressing |
| Use `breakpoint()` not `print()` | Interactive debugging; removable |
| Disable `propagate` | Prevent duplicate log entries |